# AssetWise Client — Ohio's bridge inspection data from Python

**Who this is for.** Anyone who needs Ohio bridge inventory or inspection data
without clicking through AssetWise: pull a bridge's SNBI values, its approved
inspection reports, and every inspection photo ever attached — in a few lines.

**What you need.** A `~/secrets.json` with your API credentials:

```json
{"BENTLEY_ASSETWISE_KEY_NAME": "...", "BENTLEY_ASSETWISE_API": "..."}
```

Everything in this notebook is **read-only**. The client keeps one pooled HTTPS
session (fast for many calls) and retries transient failures automatically.

**Prev:** [11. MIDAS Bridge Model Walkthrough](11.%20MIDAS%20Bridge%20Model%20Walkthrough.ipynb)
&nbsp;&middot;&nbsp; **Next:** [13. AISC Steel Sections and NSBA Splice Design ➡️](13.%20AISC%20Steel%20Sections%20and%20NSBA%20Splice%20Design.ipynb)

## Step 1 — Connect

One object, credentials loaded for you. `BASE_URL` points at ODOT's production
instance; pass a different `base_url=` for another agency's deployment.

In [ ]:
import os, sys
import pandas as pd

sys.path.insert(0, os.path.abspath("."))   # run from the civilpy repo root

from src.civilpy.state.ohio.DOT.assetwise_client import AssetWiseClient, BASE_URL

client = AssetWiseClient()
print("talking to:", BASE_URL)

## Step 2 — Find a bridge

AssetWise identifies bridges by an internal `as_id`, not the SFN you know.
`get_as_id` does the translation once — cache it if you're making many calls
(snbi_ui stores it on the bridge row for exactly this reason).

In [ ]:
SFN = "2567000"          # Scioto River pedestrian bridge, Columbus

as_id = client.get_as_id(SFN)
print(f"SFN {SFN} -> as_id {as_id}")

## Step 3 — Every current value on the bridge record

`get_current_values` returns the full field list (SNBI + Ohio agency fields) as
`fe_id`/value pairs. The `fe_id -> field name` decoder ring lives in
`civilpy.state.ohio.DOT.aw_fields`.

In [ ]:
values = client.get_current_values(as_id)
print(f"{len(values)} fields on the bridge record")
dict(list(values.items())[:3])          # {fe_id: value}

## Step 4 — Batch: many bridges, one call

For more than a handful of bridges, `get_current_values_for_assets` fetches a
whole list server-side (optionally only the `fe_ids` you care about), and
`group_values_by_asset` reshapes the flat rows into one dict per bridge.

In [ ]:
as_ids = [as_id, client.get_as_id("0100021")]

rows = client.get_current_values_for_assets(as_ids)
by_bridge = client.group_values_by_asset(rows)
{aid: f"{len(vals)} fields" for aid, vals in by_bridge.items()}

## Step 5 — Approved inspection reports

`get_inspections` lists every **approved** report on the bridge (returns `[]`
when nothing is approved yet, `None` when the asset has no report container at
all). Each row's `ast_id` is the key to everything report-scoped below.

In [ ]:
reports = client.get_inspections(as_id)
df = pd.DataFrame(reports).sort_values("ast_inspection_date", ascending=False)
cols = [c for c in ("ast_id", "ast_inspection_date", "ast_status", "ast_description")
        if c in df.columns]
df[cols].head(8)

## Step 6 — What one report actually contains

Two calls: `get_report_inspection_types` tells you which inspection type(s) the
report performed (routine / underwater / NSTM…), and `get_full_inspection_report`
returns every form field the inspector filled in.

In [ ]:
newest = max(reports, key=lambda r: r["ast_inspection_date"] or "")
ast_id = newest["ast_id"]

print("inspection types performed:", client.get_report_inspection_types(ast_id))
form = client.get_full_inspection_report(ast_id)
print(f"{len(form)} form fields on report {ast_id}")

## Step 7 — Inspection photos, including historical ones

Photos are attached **per report** and AssetWise keeps them for historical
reports — nothing is deleted or rolled forward — so walking the approved
reports recovers a bridge's complete photo history. (The asset-level file list
does *not* include these; only the per-report map surfaces them.)

Each row has the caption the inspector wrote (`af_description`) and the
download key (`af_id`).

In [ ]:
photos = client.get_report_files(ast_id)
print(f"{len(photos)} photos on the {newest['ast_inspection_date'][:10]} report")
pd.DataFrame(photos)[["af_id", "af_filename", "af_description", "af_date"]].head(6)

## Step 8 — Download a photo

`download_file(af_id)` returns the full-resolution original plus its
content-type. (`get_cover_image(as_id)` grabs the bridge's cover photo the same
way.) Below we shrink one for display.

In [ ]:
import io
from PIL import Image
from IPython.display import display

pick = next((p for p in photos if p.get("af_description")), photos[0])
data, ctype = client.download_file(pick["af_id"])
print(f"{len(data)/1e6:.1f} MB, {ctype} — {pick['af_description'] or pick['af_filename']}")

img = Image.open(io.BytesIO(data))
img.thumbnail((420, 420))
display(img)

## Step 9 — Fan-out: do anything for many bridges at once

`map_assets(fn, items)` runs your function concurrently over the pooled session
and yields `(item, result)` as they finish. One bad item yields its exception
instead of aborting the batch.

In [ ]:
sfns = ["2567000", "0100021", "2500137"]

for sfn, result in client.map_assets(client.get_as_id, sfns):
    print(f"{sfn} -> {result}")

## Step 10 — Odds and ends

* `parse_api_datetime` handles every date format the API emits (ISO 8601 and
  the legacy Microsoft `/Date(ms)/` form) and always hands back an aware UTC
  datetime.
* `iter_all_assets()` pages through the entire statewide inventory (~46k
  bridges) — use it for full syncs, not interactive work.
* `fetch_updated_assets(start_date)` is the differential-sync endpoint (assets
  modified since a date), with built-in safeguards against the server
  returning everything.
* `get_current_elements` / `get_structure_elements` pull MBEI element-level
  condition data the same way the value calls work.

In [ ]:
from src.civilpy.state.ohio.DOT.assetwise_client import parse_api_datetime

print(parse_api_datetime("2025-09-29T14:35:00"))
print(parse_api_datetime("/Date(1727620500000)/"))